##### Copyright 2026 Google LLC.

In [2]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemini API: Read a PDF

This notebook demonstrates how you can convert a PDF file so that it can be read by the Gemini API.

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/PDF_Files.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

> **Note:** This notebook uses the [Interactions API](https://ai.google.dev/gemini-api/docs/interactions), the latest way to interact with Gemini models. Looking for the `generateContent` version? Check the [archive branch](https://github.com/google-gemini/cookbook/blob/archive/generate-content-api/quickstarts/PDF_Files.ipynb).

### Setup

In [ ]:
%pip install -U -q "google-genai>=2.9.0"  # 2.0 for Interactions API

### Configure your API key

To run the following cell, your API key must be stored in a Colab Secret named `GEMINI_API_KEY`. If you don't already have an API key, or you're not sure how to create a Colab Secret, see [Authentication](https://github.com/google-gemini/cookbook/blob/main/quickstarts/Authentication.ipynb) for a walkthrough.

In [8]:
from google import genai
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=GEMINI_API_KEY)

## Download and inspect the PDF

Install the PDF processing tools. You don't need these to use the API, it's just used to display a screenshot of a page.

In [ ]:
!apt install poppler-utils

This PDF page is an article titled [Smoothly editing material properties of objects with text-to-image models and synthetic data](https://research.google/blog/smoothly-editing-material-properties-of-objects-with-text-to-image-models-and-synthetic-data/) available on the Google Research Blog.

In [13]:
import pathlib

if not pathlib.Path('test.pdf').exists():
    !curl -o test.pdf https://storage.googleapis.com/generativeai-downloads/data/Smoothly%20editing%20material%20properties%20of%20objects%20with%20text-to-image%20models%20and%20synthetic%20data.pdf

Look at one of the pages:

In [15]:
!pdftoppm test.pdf -f 1 -l 1 page-image -jpeg
!ls

In [16]:
import PIL.Image

img = PIL.Image.open(f"page-image-1.jpg")
img.thumbnail([800, 800])
img

## Read a PDF by URL

The simplest way to read a PDF is to pass its URL directly as a `document` input — no upload needed:

In [ ]:
pdf_url = "https://storage.googleapis.com/generativeai-downloads/data/Smoothly%20editing%20material%20properties%20of%20objects%20with%20text-to-image%20models%20and%20synthetic%20data.pdf"

interaction = client.interactions.create(
    model=MODEL_ID,
    input=[
        {
            "type": "document",
            "uri": pdf_url,
            "mime_type": "application/pdf",
        },
        {"type": "text", "text": "Can you summarize this file as a bulleted list?"},
    ],
)

Markdown(interaction.steps[-1].content[0].text)

> **Tip:** This works great for publicly accessible PDFs. For larger or private files, use the [File API](./File_API.ipynb) to upload them first (see below).

## Upload and read using the File API

For larger files or files you want to reuse across multiple requests, upload them using the [File API](./File_API.ipynb):

In [18]:
file_ref = client.files.upload(file='test.pdf')

Select the model you want to use in this guide:

In [20]:
MODEL_ID = "gemini-3.6-flash" # @param ["gemini-3.5-flash-lite", "gemini-2.5-flash", "gemini-3.6-flash", "gemini-2.5-pro", "gemini-3-flash-preview", "gemini-3.1-pro-preview"] {"allow-input":true, isTemplate: true}

The pages of the PDF file are each passed to the model as a screenshot of the page plus the text extracted by OCR.

In [22]:
client.models.count_tokens(
    model=MODEL_ID,
    contents=[file_ref, 'Can you summarize this file as a bulleted list?']
).total_tokens

In [23]:
interaction = client.interactions.create(
    model=MODEL_ID,
    input=[
        {"type": "document", "uri": file_ref.uri},
        {"type": "text", "text": "Can you summarize this file as a bulleted list?"},
    ],
)

In [24]:
from IPython.display import Markdown

Markdown(interaction.steps[-1].content[0].text)

In addition, take a look at how the Gemini model responds when you ask questions about the images within the PDF.

In [26]:
interaction_2 = client.interactions.create(
    model=MODEL_ID,
    input=[
        {"type": "document", "uri": file_ref.uri},
        {"type": "text", "text": "Can you explain the images and figures in this article?"},
    ],
)

In [27]:
Markdown(interaction_2.steps[-1].content[0].text)

If you observe the area of the header of the article, you can see that the model captures what is happening.

## Learning more

The File API lets you upload a variety of multimodal MIME types, including images, audio, and video formats. The File API handles inputs that can be used to generate content with `client.interactions.create` (or `client.models.generate_content`).

The File API accepts files under 2GB in size and can store up to 20GB of files per project. Files last for 2 days and cannot be downloaded from the API.

* Learn more about prompting with [media files](https://ai.google.dev/gemini-api/docs/file-prompting-strategies) in the docs, including the supported formats and maximum length.
* Learn more about to extract structured outputs from PDFs in the [Structured outputs on invoices and forms](https://github.com/google-gemini/cookbook/blob/main/examples/Pdf_structured_outputs_on_invoices_and_forms.ipynb) example.
